In [ ]:
import sys, json
sys.path.insert(0, 'src')

import torch
from riichienv import RiichiEnv
from v5model.bot import V5Bot

# 加载模型
MODEL_PATH = "artifacts/models/modelv5/best.pth"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# 创建4个agent
print("创建4个Agent...")
bots = {}
for pid in range(4):
    bots[pid] = V5Bot(
        player_id=pid,
        model_path=MODEL_PATH,
        device=DEVICE
    )
    print(f"Bot {pid} 创建完成")

print("\n开始对局...")
env = RiichiEnv(game_mode="4p-red-half")
obs_dict = env.reset()

step_count = 0
while not env.done():
    actions = {}
    for pid, obs in obs_dict.items():
        new_events = obs.new_events()
        mjai_action = None
        for evt_str in new_events:
            evt = json.loads(evt_str)
            result = bots[pid].react(evt)
            if result is not None:
                mjai_action = result
        if mjai_action is None:
            actions[pid] = obs.legal_actions()[0]
        else:
            renv_action = obs.select_action_from_mjai(json.dumps(mjai_action))
            actions[pid] = renv_action if renv_action is not None else obs.legal_actions()[0]
    obs_dict = env.step(actions)
    step_count += 1
    if step_count % 100 == 0:
        print(f"已进行 {step_count} 步, 当前得分: {env.scores()}")

print(f"\n对局结束! 共 {step_count} 步")
print(f"最终排名: {env.ranks()}")
print(f"最终得分: {env.scores()}")
agents = bots  # 兼容后续 cell 引用

In [6]:
# 获取viewer进行可视化
viewer = env.get_viewer()
print("Viewer创建成功!")
print("运行 viewer.show() 可视化对局")

Viewer创建成功!
运行 viewer.show() 可视化对局


In [7]:
# 显示对局
viewer.show(step=0, perspective=0, freeze=False)

In [ ]:
bot = agents[0]
explanation = bot.get_decision_explanation()
print(explanation)


AttributeError: 'V4Bot' object has no attribute 'get_decision_explanation'

In [5]:
# 获取决策摘要
summary = bot.tracker.generate_summary()

print("决策质量分析:")
print(f"  总决策数: {summary['total_decisions']}")
print(f"  立直决策数: {summary['riichi_count']}")
print(f"  立直率: {summary['riichi_count']/summary['total_decisions']*100:.1f}%")
print(f"  平均向听数: {summary['avg_shanten']:.2f}")
print(f"  最小向听数: {summary['min_shanten']}")

AttributeError: 'V4Bot' object has no attribute 'tracker'

In [4]:
from parse_mjlog import parse_mjlog_to_mjai
mjai_data = parse_mjlog_to_mjai("/media/bailan/DISK/AUbuntuProject/project/keqing1/dataset/tenhou6/ds1")
print(mjai_data)

KeyError: 'meta'